# AuraGateway governed full A/B/C environment qualification

Cold-session launcher for one authorized six-probe qualification. No benchmark trajectory is permitted. Use T4 x2, Internet Off, no secrets, and Save Version -> Save & Run All.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import json
import os
import socket
import sys
import traceback
import zipfile
from datetime import UTC, datetime
from pathlib import Path

NOTEBOOK_NAME = "ag-full-abc-env-qualification-v1"
INPUT_ROOT = Path("/kaggle/input").resolve()
WORK_ROOT = Path("/kaggle/working").resolve()
MATERIALIZED_HARNESS_ROOT = WORK_ROOT / "auragateway_qualification_harness"
EVIDENCE_ZIP_PATH = WORK_ROOT / "ag-qualification-evidence-v1.zip"

CONTROL_NOTEBOOK_TOKEN = "ag-qualification-control-materializer-v1"
CONTROL_OUTPUT_DIRECTORY_NAME = "ag_qualification_control_v1"
CONTROL_MANIFEST_NAME = "control_package_manifest.json"
CONTROL_RECEIPT_NAME = "materialization_receipt.json"
AUTHORIZATION_FILENAME = (
    "auragateway_full_abc_local_full_run_environment_qualification_execution_auth"
    "orization_v1.json"
)
DATASET_MANIFEST_FILENAME = "offline_dataset_manifest.json"

EXPECTED_SOURCE_MAIN_MERGE_COMMIT = "dceda98989386de7a4d57616f9f8a8023f866f10"
EXPECTED_REVIEWED_CORE_SHA256 = "6f6c7cde288d7e42810c51951d7ed4ac7952f3f47d1a6b7b05085e4f65f98763"
REVIEWED_CORE_B64 = (
    "aW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1w"
    "b3J0IHNodXRpbAppbXBvcnQgc3RhdAppbXBvcnQgc3lzCmZyb20gZGF0ZXRpbWUgaW1wb3J0IFVU"
    "QywgZGF0ZXRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpFWFBFQ1RFRF9SRVFVRVNUX1NI"
    "QTI1NiA9ICgKICAgICI3YjAwODA0MjkyNDZmNmRlZjNjMWFjMjhiOGE2NzdhMmVkN2UyOWNjZjMx"
    "ODY5MGQ5MzA5ZWQ5OGZmMTc5YmEwIgopCkVYUEVDVEVEX1JVTlRJTUVfRkFDVE9SWSA9ICgKICAg"
    "ICJhdXJhZ2F0ZXdheS5sb2NhbF9hYmMuIgogICAgImZ1bGxfYWJjX2xvY2FsX2Vudmlyb25tZW50"
    "X3F1YWxpZmljYXRpb25fa2FnZ2xlX3J1bnRpbWVfYWRhcHRlcjoiCiAgICAiY3JlYXRlX3J1bnRp"
    "bWVfYWRhcHRlciIKKQpLQUdHTEVfSU5QVVRfUk9PVCA9IFBhdGgoIi9rYWdnbGUvaW5wdXQiKS5y"
    "ZXNvbHZlKCkKS0FHR0xFX1dPUktJTkdfUk9PVCA9IFBhdGgoIi9rYWdnbGUvd29ya2luZyIpLnJl"
    "c29sdmUoKQpNQVRFUklBTElaRURfSEFSTkVTU19ST09UID0gS0FHR0xFX1dPUktJTkdfUk9PVCAv"
    "ICJhdXJhZ2F0ZXdheV9xdWFsaWZpY2F0aW9uX2hhcm5lc3MiCgoKZGVmIF9yZXF1aXJlZF9lbnZp"
    "cm9ubWVudChuYW1lKToKICAgIHZhbHVlID0gb3MuZW52aXJvbi5nZXQobmFtZSkKICAgIGlmIHZh"
    "bHVlIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYicmVxdWlyZWQgZW52aXJv"
    "bm1lbnQgYmluZGluZyBpcyBtaXNzaW5nOiB7bmFtZX0iKQogICAgcmV0dXJuIHZhbHVlCgoKZGVm"
    "IF9ib3VuZGVkX2lucHV0X3BhdGgocmF3X3BhdGgsICosIGV4cGVjdGVkX2tpbmQpOgogICAgcGF0"
    "aCA9IFBhdGgocmF3X3BhdGgpLnJlc29sdmUoKQogICAgaWYgcGF0aCAhPSBLQUdHTEVfSU5QVVRf"
    "Uk9PVCBhbmQgS0FHR0xFX0lOUFVUX1JPT1Qgbm90IGluIHBhdGgucGFyZW50czoKICAgICAgICBy"
    "YWlzZSBSdW50aW1lRXJyb3IoZiJ7ZXhwZWN0ZWRfa2luZH0gbXVzdCByZW1haW4gdW5kZXIgL2th"
    "Z2dsZS9pbnB1dCIpCiAgICByZXR1cm4gcGF0aAoKCmRlZiBfZmlsZV9zaGEyNTYocGF0aCk6CiAg"
    "ICBpZiBub3QgcGF0aC5pc19maWxlKCkgb3IgcGF0aC5pc19zeW1saW5rKCk6CiAgICAgICAgcmFp"
    "c2UgUnVudGltZUVycm9yKCJleHBlY3RlZCBvbmUgcmVndWxhciBmaWxlIikKICAgIGRpZ2VzdCA9"
    "IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggcGF0aC5vcGVuKCJyYiIpIGFzIGhhbmRsZToKICAg"
    "ICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IGhhbmRsZS5yZWFkKDEwMjQgKiAxMDI0KSwg"
    "YiIiKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBkaWdlc3Qu"
    "aGV4ZGlnZXN0KCkKCgpkZWYgX2Nhbm9uaWNhbF9zaGEyNTYocGF5bG9hZCk6CiAgICBjYW5vbmlj"
    "YWwgPSBqc29uLmR1bXBzKHBheWxvYWQsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgiLCIs"
    "ICI6IikpCiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYoY2Fub25pY2FsLmVuY29kZSgidXRmLTgi"
    "KSkuaGV4ZGlnZXN0KCkKCgpkZWYgX2RpcmVjdG9yeV9zaGEyNTYocm9vdCk6CiAgICBlbnRyaWVz"
    "ID0gW10KICAgIHRvdGFsX2J5dGVzID0gMAogICAgZm9yIHBhdGggaW4gc29ydGVkKHJvb3Qucmds"
    "b2IoIioiKSwga2V5PWxhbWJkYSBpdGVtOiBpdGVtLmFzX3Bvc2l4KCkpOgogICAgICAgIGlmIHBh"
    "dGguaXNfc3ltbGluaygpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImhhcm5lc3Mg"
    "c291cmNlIGNvbnRhaW5zIGEgc3ltYm9saWMgbGluayIpCiAgICAgICAgbWV0YWRhdGEgPSBwYXRo"
    "LnN0YXQoKQogICAgICAgIGlmIHBhdGguaXNfZGlyKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAg"
    "ICAgICAgaWYgbm90IHN0YXQuU19JU1JFRyhtZXRhZGF0YS5zdF9tb2RlKToKICAgICAgICAgICAg"
    "cmFpc2UgUnVudGltZUVycm9yKCJoYXJuZXNzIHNvdXJjZSBjb250YWlucyBhIG5vbi1yZWd1bGFy"
    "IG1lbWJlciIpCiAgICAgICAgdG90YWxfYnl0ZXMgKz0gbWV0YWRhdGEuc3Rfc2l6ZQogICAgICAg"
    "IGlmIHRvdGFsX2J5dGVzID4gMTAwICogMTAyNCAqIDEwMjQ6CiAgICAgICAgICAgIHJhaXNlIFJ1"
    "bnRpbWVFcnJvcigiaGFybmVzcyBzb3VyY2UgZXhjZWVkcyB0aGUgYm9vdHN0cmFwIGJ5dGUgYnVk"
    "Z2V0IikKICAgICAgICBlbnRyaWVzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAg"
    "ICAgInBhdGgiOiBwYXRoLnJlbGF0aXZlX3RvKHJvb3QpLmFzX3Bvc2l4KCksCiAgICAgICAgICAg"
    "ICAgICAic2hhMjU2IjogX2ZpbGVfc2hhMjU2KHBhdGgpLAogICAgICAgICAgICAgICAgInNpemVf"
    "Ynl0ZXMiOiBtZXRhZGF0YS5zdF9zaXplLAogICAgICAgICAgICB9CiAgICAgICAgKQogICAgICAg"
    "IGlmIGxlbihlbnRyaWVzKSA+IDVfMDAwOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3Io"
    "Imhhcm5lc3Mgc291cmNlIGV4Y2VlZHMgdGhlIGJvb3RzdHJhcCBmaWxlIGJ1ZGdldCIpCiAgICBp"
    "ZiBub3QgZW50cmllczoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImhhcm5lc3Mgc291cmNl"
    "IGlzIGVtcHR5IikKICAgIHJldHVybiBfY2Fub25pY2FsX3NoYTI1NihlbnRyaWVzKQoKCmRlZiBf"
    "cGFyc2VfdGltZXN0YW1wKHJhd192YWx1ZSk6CiAgICBpZiBub3QgaXNpbnN0YW5jZShyYXdfdmFs"
    "dWUsIHN0cik6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJhdXRob3JpemF0aW9uIHRpbWVz"
    "dGFtcCBpcyBpbnZhbGlkIikKICAgIG5vcm1hbGl6ZWQgPSByYXdfdmFsdWUucmVwbGFjZSgiWiIs"
    "ICIrMDA6MDAiKQogICAgdmFsdWUgPSBkYXRldGltZS5mcm9taXNvZm9ybWF0KG5vcm1hbGl6ZWQp"
    "CiAgICBpZiB2YWx1ZS50emluZm8gaXMgTm9uZSBvciB2YWx1ZS51dGNvZmZzZXQoKSBpcyBOb25l"
    "OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiYXV0aG9yaXphdGlvbiB0aW1lc3RhbXAgbXVz"
    "dCBiZSB0aW1lem9uZS1hd2FyZSIpCiAgICByZXR1cm4gdmFsdWUKCgphdXRob3JpemF0aW9uX3Bh"
    "dGggPSBfYm91bmRlZF9pbnB1dF9wYXRoKAogICAgX3JlcXVpcmVkX2Vudmlyb25tZW50KCJBVVJB"
    "R0FURVdBWV9RVUFMSUZJQ0FUSU9OX0FVVEhPUklaQVRJT04iKSwKICAgIGV4cGVjdGVkX2tpbmQ9"
    "ImF1dGhvcml6YXRpb24iLAopCm1hbmlmZXN0X3BhdGggPSBfYm91bmRlZF9pbnB1dF9wYXRoKAog"
    "ICAgX3JlcXVpcmVkX2Vudmlyb25tZW50KCJBVVJBR0FURVdBWV9RVUFMSUZJQ0FUSU9OX0RBVEFT"
    "RVRfTUFOSUZFU1QiKSwKICAgIGV4cGVjdGVkX2tpbmQ9ImRhdGFzZXQgbWFuaWZlc3QiLAopCmF1"
    "dGhvcml6YXRpb24gPSBqc29uLmxvYWRzKGF1dGhvcml6YXRpb25fcGF0aC5yZWFkX3RleHQoZW5j"
    "b2Rpbmc9InV0Zi04IikpCm1hbmlmZXN0ID0ganNvbi5sb2FkcyhtYW5pZmVzdF9wYXRoLnJlYWRf"
    "dGV4dChlbmNvZGluZz0idXRmLTgiKSkKaWYgYXV0aG9yaXphdGlvbi5nZXQoImRlY2lzaW9uIikg"
    "IT0gIkFVVEhPUklaRUQiOgogICAgcmFpc2UgUnVudGltZUVycm9yKCJhdXRob3JpemF0aW9uIGRl"
    "Y2lzaW9uIGlzIG5vdCBBVVRIT1JJWkVEIikKaWYgYXV0aG9yaXphdGlvbi5nZXQoInJlcXVlc3Rf"
    "c2hhMjU2IikgIT0gRVhQRUNURURfUkVRVUVTVF9TSEEyNTY6CiAgICByYWlzZSBSdW50aW1lRXJy"
    "b3IoImF1dGhvcml6YXRpb24gZG9lcyBub3QgYmluZCB0aGUgZXhwZWN0ZWQgcmVxdWVzdCIpCmlm"
    "IGF1dGhvcml6YXRpb24uZ2V0KCJkYXRhc2V0X21hbmlmZXN0X3NoYTI1NiIpICE9IF9jYW5vbmlj"
    "YWxfc2hhMjU2KG1hbmlmZXN0KToKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiYXV0aG9yaXphdGlv"
    "biBkb2VzIG5vdCBiaW5kIHRoZSBzdXBwbGllZCBkYXRhc2V0IG1hbmlmZXN0IikKZW50cmllcyA9"
    "IG1hbmlmZXN0LmdldCgiZW50cmllcyIpCmV4cGVjdGVkX2VudHJ5X3NoYXBlcyA9ICgKICAgICgi"
    "aGFybmVzc19zb3VyY2UiLCAic291cmNlX3RyZWVfZGlyZWN0b3J5IiksCiAgICAoIm1vZGVsX2Fy"
    "dGlmYWN0cyIsICJodWdnaW5nX2ZhY2Vfc25hcHNob3RfZGlyZWN0b3J5IiksCiAgICAoInZsbG1f"
    "cnVudGltZSIsICJweXRob25fd2hlZWxob3VzZV9kaXJlY3RvcnkiKSwKKQppZiBub3QgaXNpbnN0"
    "YW5jZShlbnRyaWVzLCBsaXN0KSBvciBsZW4oZW50cmllcykgIT0gbGVuKGV4cGVjdGVkX2VudHJ5"
    "X3NoYXBlcyk6CiAgICByYWlzZSBSdW50aW1lRXJyb3IoImRhdGFzZXQgbWFuaWZlc3QgZW50cnkg"
    "c2V0IGRyaWZ0ZWQiKQpmb3IgZW50cnksIGV4cGVjdGVkX3NoYXBlIGluIHppcChlbnRyaWVzLCBl"
    "eHBlY3RlZF9lbnRyeV9zaGFwZXMsIHN0cmljdD1UcnVlKToKICAgIGlmIG5vdCBpc2luc3RhbmNl"
    "KGVudHJ5LCBkaWN0KToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImRhdGFzZXQgbWFuaWZl"
    "c3QgZW50cnkgaXMgaW52YWxpZCIpCiAgICBvYnNlcnZlZF9zaGFwZSA9IChlbnRyeS5nZXQoInJv"
    "bGUiKSwgZW50cnkuZ2V0KCJhcnRpZmFjdF9mb3JtYXQiKSkKICAgIGlmIG9ic2VydmVkX3NoYXBl"
    "ICE9IGV4cGVjdGVkX3NoYXBlOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiZGF0YXNldCBt"
    "YW5pZmVzdCByb2xlIG9yIGFydGlmYWN0IGZvcm1hdCBkcmlmdGVkIikKICAgIGlmIGV4cGVjdGVk"
    "X3NoYXBlWzBdICE9ICJ2bGxtX3J1bnRpbWUiOgogICAgICAgIF9ib3VuZGVkX2lucHV0X3BhdGgo"
    "CiAgICAgICAgICAgIGVudHJ5LmdldCgibW91bnRlZF9wYXRoIiksCiAgICAgICAgICAgIGV4cGVj"
    "dGVkX2tpbmQ9ZiJ7ZXhwZWN0ZWRfc2hhcGVbMF19IGlucHV0IiwKICAgICAgICApCiAgICBlbGlm"
    "IGVudHJ5LmdldCgibW91bnRlZF9wYXRoIikgaXMgbm90IE5vbmU6CiAgICAgICAgcmFpc2UgUnVu"
    "dGltZUVycm9yKCJ2TExNIHJ1bnRpbWUgbXVzdCB1c2UgYm91bmRlZCBvdXRwdXQtZGlyZWN0b3J5"
    "IGRpc2NvdmVyeSIpCmlmIGFueSgKICAgICgKICAgICAgICBtYW5pZmVzdC5nZXQoIm5ldHdvcmtf"
    "YWNjZXNzX3Blcm1pdHRlZCIpIGlzIG5vdCBGYWxzZSwKICAgICAgICBtYW5pZmVzdC5nZXQoImNy"
    "ZWRlbnRpYWxzX3ByZXNlbnQiKSBpcyBub3QgRmFsc2UsCiAgICAgICAgbWFuaWZlc3QuZ2V0KCJj"
    "dXN0b21lcl9kYXRhX3ByZXNlbnQiKSBpcyBub3QgRmFsc2UsCiAgICAgICAgbWFuaWZlc3QuZ2V0"
    "KCJob3N0ZWRfcHJvdmlkZXJfaW5wdXRzX3ByZXNlbnQiKSBpcyBub3QgRmFsc2UsCiAgICApCik6"
    "CiAgICByYWlzZSBSdW50aW1lRXJyb3IoImRhdGFzZXQgbWFuaWZlc3Qgc2FmZXR5IGVudmVsb3Bl"
    "IGRyaWZ0ZWQiKQppZiBhbnkoCiAgICAoCiAgICAgICAgYXV0aG9yaXphdGlvbi5nZXQoIm1heGlt"
    "dW1fa2FnZ2xlX3Nlc3Npb25zIikgIT0gMSwKICAgICAgICBhdXRob3JpemF0aW9uLmdldCgibWF4"
    "aW11bV9tb2RlbF9yZXF1ZXN0cyIpICE9IDgsCiAgICAgICAgYXV0aG9yaXphdGlvbi5nZXQoIm1h"
    "eGltdW1fb3V0cHV0X3Rva2Vuc19wZXJfcmVxdWVzdCIpICE9IDMyLAogICAgICAgIGF1dGhvcml6"
    "YXRpb24uZ2V0KCJiZW5jaG1hcmtfdHJhamVjdG9yeV9yZXF1ZXN0c19wZXJtaXR0ZWQiKSAhPSAw"
    "LAogICAgICAgIGF1dGhvcml6YXRpb24uZ2V0KCJjdXN0b21lcl9kYXRhX3Blcm1pdHRlZCIpIGlz"
    "IG5vdCBGYWxzZSwKICAgICAgICBhdXRob3JpemF0aW9uLmdldCgiY3JlZGVudGlhbHNfcGVybWl0"
    "dGVkIikgaXMgbm90IEZhbHNlLAogICAgICAgIGF1dGhvcml6YXRpb24uZ2V0KCJuZXR3b3JrX2Fj"
    "Y2Vzc19wZXJtaXR0ZWQiKSBpcyBub3QgRmFsc2UsCiAgICAgICAgYXV0aG9yaXphdGlvbi5nZXQo"
    "ImV4dGVybmFsX3NwZW5kIikgIT0gMCwKICAgICAgICBhdXRob3JpemF0aW9uLmdldCgibWVhc3Vy"
    "ZWRfZXhlY3V0aW9uX2F1dGhvcml6ZWQiKSBpcyBub3QgRmFsc2UsCiAgICApCik6CiAgICByYWlz"
    "ZSBSdW50aW1lRXJyb3IoImF1dGhvcml6YXRpb24gc2FmZXR5IGVudmVsb3BlIGRyaWZ0ZWQiKQpu"
    "b3cgPSBkYXRldGltZS5ub3coVVRDKQppc3N1ZWRfYXQgPSBfcGFyc2VfdGltZXN0YW1wKGF1dGhv"
    "cml6YXRpb24uZ2V0KCJpc3N1ZWRfYXQiKSkKZXhwaXJlc19hdCA9IF9wYXJzZV90aW1lc3RhbXAo"
    "YXV0aG9yaXphdGlvbi5nZXQoImV4cGlyZXNfYXQiKSkKaWYgbm90IGlzc3VlZF9hdCA8PSBub3cg"
    "PCBleHBpcmVzX2F0OgogICAgcmFpc2UgUnVudGltZUVycm9yKCJhdXRob3JpemF0aW9uIGlzIG91"
    "dHNpZGUgaXRzIHZhbGlkaXR5IHdpbmRvdyIpCgpoYXJuZXNzX2VudHJpZXMgPSBbZW50cnkgZm9y"
    "IGVudHJ5IGluIGVudHJpZXMgaWYgZW50cnkuZ2V0KCJyb2xlIikgPT0gImhhcm5lc3Nfc291cmNl"
    "Il0KaWYgbGVuKGhhcm5lc3NfZW50cmllcykgIT0gMToKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigi"
    "ZGF0YXNldCBtYW5pZmVzdCBtdXN0IGNvbnRhaW4gb25lIGhhcm5lc3Nfc291cmNlIGVudHJ5IikK"
    "aGFybmVzc19lbnRyeSA9IGhhcm5lc3NfZW50cmllc1swXQppZiBoYXJuZXNzX2VudHJ5LmdldCgi"
    "YXJ0aWZhY3RfZm9ybWF0IikgIT0gInNvdXJjZV90cmVlX2RpcmVjdG9yeSI6CiAgICByYWlzZSBS"
    "dW50aW1lRXJyb3IoImhhcm5lc3Nfc291cmNlIG11c3QgdXNlIHNvdXJjZV90cmVlX2RpcmVjdG9y"
    "eSIpCm1vdW50ZWRfcmVwb19yb290ID0gX2JvdW5kZWRfaW5wdXRfcGF0aCgKICAgIGhhcm5lc3Nf"
    "ZW50cnkuZ2V0KCJtb3VudGVkX3BhdGgiKSwKICAgIGV4cGVjdGVkX2tpbmQ9Imhhcm5lc3Mgc291"
    "cmNlIiwKKQppZiBub3QgbW91bnRlZF9yZXBvX3Jvb3QuaXNfZGlyKCkgb3IgbW91bnRlZF9yZXBv"
    "X3Jvb3QuaXNfc3ltbGluaygpOgogICAgcmFpc2UgUnVudGltZUVycm9yKCJoYXJuZXNzX3NvdXJj"
    "ZSBtdXN0IHJlc29sdmUgdG8gb25lIHJlYWwgZGlyZWN0b3J5IikKaGFybmVzc19zaGEyNTYgPSBo"
    "YXJuZXNzX2VudHJ5LmdldCgic2hhMjU2IikKaWYgX2RpcmVjdG9yeV9zaGEyNTYobW91bnRlZF9y"
    "ZXBvX3Jvb3QpICE9IGhhcm5lc3Nfc2hhMjU2OgogICAgcmFpc2UgUnVudGltZUVycm9yKCJoYXJu"
    "ZXNzX3NvdXJjZSBpZGVudGl0eSBkb2VzIG5vdCBtYXRjaCB0aGUgbWFuaWZlc3QiKQptb3VudGVk"
    "X3NvdXJjZV9yb290ID0gbW91bnRlZF9yZXBvX3Jvb3QgLyAic3JjIgpyZXF1aXJlZF9oYXJuZXNz"
    "X3BhdGhzID0gKAogICAgbW91bnRlZF9yZXBvX3Jvb3QgLyAicHlwcm9qZWN0LnRvbWwiLAogICAg"
    "bW91bnRlZF9zb3VyY2Vfcm9vdCAvICJhdXJhZ2F0ZXdheSIsCiAgICBtb3VudGVkX3JlcG9fcm9v"
    "dCAvICJkYXRhL2V2YWxzL2JlbmNobWFyay9lbnZpcm9ubWVudC1xdWFsaWZpY2F0aW9uLXYxLyIK"
    "ICAgICJxdWFsaWZpY2F0aW9uX2V4ZWN1dGlvbl9yZXF1ZXN0Lmpzb24iLAopCmlmIG5vdCByZXF1"
    "aXJlZF9oYXJuZXNzX3BhdGhzWzBdLmlzX2ZpbGUoKToKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigi"
    "aGFybmVzc19zb3VyY2UgZG9lcyBub3QgY29udGFpbiBweXByb2plY3QudG9tbCIpCmlmIG5vdCBy"
    "ZXF1aXJlZF9oYXJuZXNzX3BhdGhzWzFdLmlzX2RpcigpOgogICAgcmFpc2UgUnVudGltZUVycm9y"
    "KCJoYXJuZXNzX3NvdXJjZSBkb2VzIG5vdCBjb250YWluIHNyYy9hdXJhZ2F0ZXdheSIpCmlmIG5v"
    "dCByZXF1aXJlZF9oYXJuZXNzX3BhdGhzWzJdLmlzX2ZpbGUoKToKICAgIHJhaXNlIFJ1bnRpbWVF"
    "cnJvcigiaGFybmVzc19zb3VyY2UgZG9lcyBub3QgY29udGFpbiB0aGUgZXhlY3V0aW9uIHJlcXVl"
    "c3QiKQppZiBub3QgS0FHR0xFX1dPUktJTkdfUk9PVC5pc19kaXIoKToKICAgIHJhaXNlIFJ1bnRp"
    "bWVFcnJvcigiL2thZ2dsZS93b3JraW5nIGlzIHVuYXZhaWxhYmxlIikKaWYgTUFURVJJQUxJWkVE"
    "X0hBUk5FU1NfUk9PVC5leGlzdHMoKToKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigid3JpdGFibGUg"
    "aGFybmVzcyBkZXN0aW5hdGlvbiBhbHJlYWR5IGV4aXN0cyIpCnNodXRpbC5jb3B5dHJlZShtb3Vu"
    "dGVkX3JlcG9fcm9vdCwgTUFURVJJQUxJWkVEX0hBUk5FU1NfUk9PVCkKcmVwb19yb290ID0gTUFU"
    "RVJJQUxJWkVEX0hBUk5FU1NfUk9PVC5yZXNvbHZlKCkKaWYgX2RpcmVjdG9yeV9zaGEyNTYocmVw"
    "b19yb290KSAhPSBoYXJuZXNzX3NoYTI1NjoKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigid3JpdGFi"
    "bGUgaGFybmVzcyBjb3B5IGlkZW50aXR5IGRyaWZ0ZWQiKQpzb3VyY2Vfcm9vdCA9IHJlcG9fcm9v"
    "dCAvICJzcmMiCgpydW50aW1lX2ZhY3RvcnkgPSBhdXRob3JpemF0aW9uLmdldCgicnVudGltZV9m"
    "YWN0b3J5IikKaWYgbm90IGlzaW5zdGFuY2UocnVudGltZV9mYWN0b3J5LCBkaWN0KToKICAgIHJh"
    "aXNlIFJ1bnRpbWVFcnJvcigiYXV0aG9yaXphdGlvbiBydW50aW1lIGZhY3RvcnkgaXMgaW52YWxp"
    "ZCIpCmlmIHJ1bnRpbWVfZmFjdG9yeS5nZXQoImZhY3RvcnlfcGF0aCIpICE9IEVYUEVDVEVEX1JV"
    "TlRJTUVfRkFDVE9SWToKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiYXV0aG9yaXphdGlvbiBydW50"
    "aW1lIGZhY3RvcnkgcGF0aCBkcmlmdGVkIikKYWRhcHRlcl9yZWxhdGl2ZV9wYXRoID0gUGF0aChy"
    "dW50aW1lX2ZhY3RvcnkuZ2V0KCJhcnRpZmFjdF9wYXRoIiwgIiIpKQppZiBhZGFwdGVyX3JlbGF0"
    "aXZlX3BhdGguaXNfYWJzb2x1dGUoKSBvciAiLi4iIGluIGFkYXB0ZXJfcmVsYXRpdmVfcGF0aC5w"
    "YXJ0czoKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigicnVudGltZSBhZGFwdGVyIHBhdGggbXVzdCBi"
    "ZSByZXBvc2l0b3J5LXJlbGF0aXZlIikKYWRhcHRlcl9wYXRoID0gKHJlcG9fcm9vdCAvIGFkYXB0"
    "ZXJfcmVsYXRpdmVfcGF0aCkucmVzb2x2ZSgpCmlmIGFkYXB0ZXJfcGF0aCAhPSByZXBvX3Jvb3Qg"
    "YW5kIHJlcG9fcm9vdCBub3QgaW4gYWRhcHRlcl9wYXRoLnBhcmVudHM6CiAgICByYWlzZSBSdW50"
    "aW1lRXJyb3IoInJ1bnRpbWUgYWRhcHRlciBtdXN0IHJlbWFpbiBpbnNpZGUgaGFybmVzc19zb3Vy"
    "Y2UiKQppZiBfZmlsZV9zaGEyNTYoYWRhcHRlcl9wYXRoKSAhPSBydW50aW1lX2ZhY3RvcnkuZ2V0"
    "KCJhcnRpZmFjdF9zaGEyNTYiKToKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigicnVudGltZSBhZGFw"
    "dGVyIGlkZW50aXR5IGRvZXMgbm90IG1hdGNoIGF1dGhvcml6YXRpb24iKQoKb3MuZW52aXJvblsi"
    "QVVSQUdBVEVXQVlfUkVQT19ST09UIl0gPSBzdHIocmVwb19yb290KQpzeXMucGF0aC5pbnNlcnQo"
    "MCwgc3RyKHNvdXJjZV9yb290KSkKCmV4ZWN1dGlvbl9tb2R1bGUgPSBpbXBvcnRsaWIuaW1wb3J0"
    "X21vZHVsZSgKICAgICJhdXJhZ2F0ZXdheS5sb2NhbF9hYmMuZnVsbF9hYmNfbG9jYWxfZW52aXJv"
    "bm1lbnRfcXVhbGlmaWNhdGlvbl9leGVjdXRpb24iCikKc3VtbWFyeSA9IGV4ZWN1dGlvbl9tb2R1"
    "bGUuZXhlY3V0ZV9mcm9tX2Vudmlyb25tZW50KCkKc3VtbWFyeQ=="
)

EXPECTED_HARNESS_SOURCE = Path(
    "/kaggle/input/notebooks/kabomolefe/ag-worker-obs-harness-materializer-v1/ag_"
    "worker_obs_harness_materializer_v1_output/auragateway_qualification_harness_"
    "dceda98_worker_obs_v1"
)
EXPECTED_MODEL_SNAPSHOT = Path(
    "/kaggle/input/datasets/kabomolefe/auragateway-qwen2-5-0-5b-offline-v1/auraga"
    "teway-qwen2.5-0.5b-instruct-7ae557604adf67be50417f59c2c2f167def9a775/hf_home"
    "/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c"
    "2c2f167def9a775"
)
EXPECTED_RUNTIME_OUTPUT_DIRECTORY = "auragateway_vllm_cu129_wheelhouse_v1"
EXPECTED_RUNTIME_RESOLUTION_LOCK_SHA256 = (
    "1575538b0a412c9b030fc95ccada0f0527553b76f06ef6b2b72904e61c84870c"
)
EXPECTED_RUNTIME_MANIFEST_SHA256 = (
    "b424d2b952d726b2f7451ebd8f48d604985f650dbe2f6d146969625618b7fc51"
)
EXPECTED_RUNTIME_SHA256_MANIFEST_SHA256 = (
    "789fb23ab7d9c4f28dd909e808a53a65d692c0d7b43bc44da9e974817d771b8d"
)
EXPECTED_RUNTIME_MATERIALIZATION_RECEIPT_SHA256 = (
    "52aa42b940dd606ab5685686ab893eb085efed2a7466989f654e870f4b360589"
)
EXPECTED_RUNTIME_PACKAGE_COUNT = 176

MINIMUM_LAUNCH_WINDOW_MINUTES = 120
MAXIMUM_EVIDENCE_ZIP_BYTES = 2097152
WORKER_STARTUP_DIAGNOSTIC_RELATIVE_PATH = (
    "data/evals/benchmark/environment-qualification-v1/worker_startup_diagnostic.json"
)
MAXIMUM_WORKER_STARTUP_DIAGNOSTIC_BYTES = (
    262144
)

RUNTIME_EVIDENCE_PATHS = (
    "data/evals/benchmark/environment-qualification-v1/cache_metric_capability_report.json",
    "data/evals/benchmark/environment-qualification-v1/gpu_topology_report.json",
    "data/evals/benchmark/environment-qualification-v1/kaggle_runtime_dependency_lock.json",
    "data/evals/benchmark/environment-qualification-v1/manifest.json",
    "data/evals/benchmark/environment-qualification-v1/model_identity_report.json",
    "data/evals/benchmark/environment-qualification-v1/qualification_report.json",
    "data/evals/benchmark/environment-qualification-v1/reset_capability_report.json",
    "data/evals/benchmark/environment-qualification-v1/worker_health_report.json",
)

stage = "launcher_initialization"


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def parse_timestamp(raw_value: object) -> datetime:
    if not isinstance(raw_value, str):
        raise RuntimeError("authorization timestamp is invalid")
    value = datetime.fromisoformat(raw_value.replace("Z", "+00:00"))
    if value.tzinfo is None or value.utcoffset() is None:
        raise RuntimeError("authorization timestamp must be timezone-aware")
    return value.astimezone(UTC)


def port_open(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1.0):
            return True
    except OSError:
        return False


def resolve_control_output() -> tuple[Path, Path, Path, Path]:
    candidate_roots = tuple(
        sorted(
            (
                path.resolve()
                for path in INPUT_ROOT.rglob(CONTROL_OUTPUT_DIRECTORY_NAME)
                if path.is_dir()
                and not path.is_symlink()
                and CONTROL_NOTEBOOK_TOKEN in path.resolve().as_posix()
            ),
            key=lambda item: item.as_posix(),
        )
    )
    relative_candidates = tuple(
        path.relative_to(INPUT_ROOT).as_posix()
        for path in candidate_roots
        if INPUT_ROOT in path.parents
    )
    if len(candidate_roots) != 1:
        raise RuntimeError(
            "expected exactly one governed control-output root; "
            f"observed={len(candidate_roots)}; "
            f"candidates={relative_candidates}"
        )

    control_root = candidate_roots[0]
    if INPUT_ROOT not in control_root.parents:
        raise RuntimeError("control output escaped /kaggle/input")

    expected_control_names = {
        AUTHORIZATION_FILENAME,
        DATASET_MANIFEST_FILENAME,
        CONTROL_MANIFEST_NAME,
        CONTROL_RECEIPT_NAME,
    }
    control_members = tuple(
        sorted(
            control_root.iterdir(),
            key=lambda item: item.name,
        )
    )
    observed_control_names = {path.name for path in control_members}
    if observed_control_names != expected_control_names:
        raise RuntimeError(
            "control output file set drifted; "
            f"expected={tuple(sorted(expected_control_names))}; "
            f"observed={tuple(sorted(observed_control_names))}"
        )

    for path in control_members:
        if path.is_symlink() or not path.is_file():
            raise RuntimeError("control output contains an unsafe member type")
        if path.suffix.lower() in {".zip", ".tar", ".tgz", ".7z"}:
            raise RuntimeError("control output cannot contain nested archives")
        validate_regular_file(path, maximum_bytes=1024 * 1024)

    return (
        control_root / AUTHORIZATION_FILENAME,
        control_root / DATASET_MANIFEST_FILENAME,
        control_root / CONTROL_MANIFEST_NAME,
        control_root / CONTROL_RECEIPT_NAME,
    )


def validate_regular_file(path: Path, *, maximum_bytes: int) -> None:
    if not path.is_file() or path.is_symlink():
        raise RuntimeError(f"expected one regular file: {path.name}")
    if path.stat().st_size > maximum_bytes:
        raise RuntimeError(f"bounded file exceeds its size limit: {path.name}")


def load_worker_startup_diagnostic() -> bytes | None:
    path = (
        MATERIALIZED_HARNESS_ROOT
        / WORKER_STARTUP_DIAGNOSTIC_RELATIVE_PATH
    )
    if not path.exists():
        return None
    validate_regular_file(
        path,
        maximum_bytes=MAXIMUM_WORKER_STARTUP_DIAGNOSTIC_BYTES,
    )
    raw = path.read_bytes()
    try:
        payload = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise RuntimeError(
            "worker-startup diagnostic is invalid JSON"
        ) from error
    if not isinstance(payload, dict):
        raise RuntimeError("worker-startup diagnostic root is invalid")
    expected = {
        "schema_version": "1.0.0",
        "diagnostic_id": (
            "auragateway-environment-qualification-"
            "worker-startup-diagnostic-v1"
        ),
        "status": "FAILED",
        "raw_environment_included": False,
        "authorization_payload_included": False,
        "model_content_included": False,
        "hidden_retries_performed": 0,
        "workers_replaced": 0,
        "model_requests_performed": 0,
        "benchmark_trajectory_requests_performed": 0,
    }
    if any(payload.get(key) != value for key, value in expected.items()):
        raise RuntimeError("worker-startup diagnostic safety drifted")
    workers = payload.get("workers")
    if not isinstance(workers, list) or len(workers) != 2:
        raise RuntimeError("worker-startup diagnostic worker set drifted")
    identities = tuple(
        worker.get("worker_id")
        for worker in workers
        if isinstance(worker, dict)
    )
    if identities != ("worker_1", "worker_2"):
        raise RuntimeError("worker-startup diagnostic identity drifted")
    prohibited_keys = {
        "environment",
        "env",
        "authorization_payload",
        "model_content",
        "command_argv",
    }

    def reject_prohibited_keys(value: object) -> None:
        if isinstance(value, dict):
            if prohibited_keys.intersection(value):
                raise RuntimeError(
                    "worker-startup diagnostic contains prohibited fields"
                )
            for item in value.values():
                reject_prohibited_keys(item)
        elif isinstance(value, list):
            for item in value:
                reject_prohibited_keys(item)

    reject_prohibited_keys(payload)
    if raw != canonical_json(payload).encode("utf-8"):
        raise RuntimeError("worker-startup diagnostic is not canonical JSON")
    return raw


def write_failure_bundle(error: BaseException) -> None:
    evidence_found: list[str] = []
    if MATERIALIZED_HARNESS_ROOT.is_dir():
        for relative_path in RUNTIME_EVIDENCE_PATHS:
            path = MATERIALIZED_HARNESS_ROOT / relative_path
            if path.is_file() and not path.is_symlink():
                evidence_found.append(relative_path)

    diagnostic_bytes = load_worker_startup_diagnostic()
    failure = {
        "schema_version": "1.0.0",
        "status": "FAILED",
        "stage": stage,
        "captured_at": datetime.now(UTC).replace(microsecond=0).isoformat(),
        "exception_type": type(error).__name__,
        "safe_message": str(error)[:1000],
        "runtime_evidence_found": sorted(evidence_found),
        "worker_startup_diagnostic_included": (
            diagnostic_bytes is not None
        ),
        "ports_open": [
            port
            for port in (8001, 8002)
            if port_open(port)
        ],
        "benchmark_trajectory_requests_permitted": 0,
        "customer_data_used": False,
        "credentials_used": False,
        "provider_calls_performed": False,
        "external_spend": 0,
    }
    trace = "".join(
        traceback.format_exception(type(error), error, error.__traceback__)
    )[-65536:]

    if EVIDENCE_ZIP_PATH.exists():
        EVIDENCE_ZIP_PATH.unlink()

    with zipfile.ZipFile(
        EVIDENCE_ZIP_PATH,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        archive.writestr(
            "launcher_failure.json",
            canonical_json(failure),
        )
        archive.writestr(
            "launcher_failure_trace.txt",
            trace,
        )
        if diagnostic_bytes is not None:
            archive.writestr(
                "worker_startup_diagnostic.json",
                diagnostic_bytes,
            )

    if EVIDENCE_ZIP_PATH.stat().st_size > MAXIMUM_EVIDENCE_ZIP_BYTES:
        EVIDENCE_ZIP_PATH.unlink(missing_ok=True)
        raise RuntimeError("failure evidence ZIP exceeded the size budget")

    print(f"artifact={EVIDENCE_ZIP_PATH}")
    print(f"size_bytes={EVIDENCE_ZIP_PATH.stat().st_size}")
    print(f"sha256={file_sha256(EVIDENCE_ZIP_PATH)}")
    print("qualification_status=FAILED")
    print(
        "worker_startup_diagnostic_included="
        + str(diagnostic_bytes is not None).lower()
    )
    print("upload_only_this_file=true")


try:
    stage = "fresh_session_guard"

    if not INPUT_ROOT.is_dir() or not WORK_ROOT.is_dir():
        raise RuntimeError("required Kaggle filesystem roots are unavailable")

    if len(NOTEBOOK_NAME) > 50:
        raise RuntimeError("Kaggle notebook name exceeds 50 characters")

    loaded_runtime_modules = sorted(
        name
        for name in sys.modules
        if name == "vllm"
        or name.startswith("vllm.")
        or name == "transformers"
        or name.startswith("transformers.")
    )
    if loaded_runtime_modules:
        raise RuntimeError(
            "qualification requires a fresh kernel without loaded runtime modules: "
            + ", ".join(loaded_runtime_modules)
        )

    if "torch" in sys.modules:
        torch_module = sys.modules["torch"]
        cuda_module = getattr(torch_module, "cuda", None)
        is_initialized = getattr(cuda_module, "is_initialized", None)
        if callable(is_initialized) and bool(is_initialized()):
            raise RuntimeError("qualification found a pre-existing CUDA context")

    stale_runtime_keys = sorted(
        key
        for key in os.environ
        if key.startswith("AURAGATEWAY_") or key.startswith("VLLM_")
    )
    if stale_runtime_keys:
        raise RuntimeError(
            "qualification found stale AuraGateway/vLLM environment variables: "
            + ", ".join(stale_runtime_keys)
        )

    if MATERIALIZED_HARNESS_ROOT.exists():
        raise RuntimeError("writable harness destination already exists")

    if EVIDENCE_ZIP_PATH.exists():
        raise RuntimeError("qualification evidence ZIP already exists")

    open_ports = [port for port in (8001, 8002) if port_open(port)]
    if open_ports:
        raise RuntimeError(f"vLLM worker ports are already open: {open_ports}")

    for expected_path in (
        EXPECTED_HARNESS_SOURCE,
        EXPECTED_MODEL_SNAPSHOT,
    ):
        resolved = expected_path.resolve()
        if INPUT_ROOT not in resolved.parents:
            raise RuntimeError("a static input escaped /kaggle/input")
        if not resolved.exists() or resolved.is_symlink():
            raise RuntimeError(f"expected static input is unavailable: {resolved}")

    stage = "control_output_discovery"

    (
        authorization_path,
        dataset_manifest_path,
        control_manifest_path,
        receipt_path,
    ) = resolve_control_output()

    control_manifest = json.loads(
        control_manifest_path.read_text(encoding="utf-8")
    )
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    authorization = json.loads(
        authorization_path.read_text(encoding="utf-8")
    )
    dataset_manifest = json.loads(
        dataset_manifest_path.read_text(encoding="utf-8")
    )
    if not all(
        isinstance(payload, dict)
        for payload in (
            control_manifest,
            receipt,
            authorization,
            dataset_manifest,
        )
    ):
        raise RuntimeError("control output JSON roots must be objects")

    if control_manifest.get("source_main_merge_commit") != (
        EXPECTED_SOURCE_MAIN_MERGE_COMMIT
    ):
        raise RuntimeError("control output source main binding drifted")
    authorization_source_main_merge_commit = authorization.get(
        "source_main_merge_commit"
    )
    if (
        not isinstance(authorization_source_main_merge_commit, str)
        or len(authorization_source_main_merge_commit) != 40
        or any(
            character not in "0123456789abcdef"
            for character in authorization_source_main_merge_commit
        )
    ):
        raise RuntimeError("authorization source main binding is invalid")
    if control_manifest.get("authorization_source_main_merge_commit") != (
        authorization_source_main_merge_commit
    ):
        raise RuntimeError("authorization source main binding drifted")

    if control_manifest.get("authorization_file_sha256") != file_sha256(
        authorization_path
    ):
        raise RuntimeError("authorization file identity drifted")
    if control_manifest.get("dataset_manifest_file_sha256") != file_sha256(
        dataset_manifest_path
    ):
        raise RuntimeError("dataset manifest file identity drifted")

    if receipt.get("control_manifest_sha256") != file_sha256(
        control_manifest_path
    ):
        raise RuntimeError("control materialization receipt drifted")

    if authorization.get("decision") != "AUTHORIZED":
        raise RuntimeError("authorization decision is not AUTHORIZED")

    if any(
        (
            authorization.get("maximum_workers") != 2,
            authorization.get("maximum_kaggle_sessions") != 1,
            authorization.get("maximum_model_requests") != 8,
            authorization.get("maximum_output_tokens_per_request") != 32,
            authorization.get("benchmark_trajectory_requests_permitted") != 0,
            authorization.get("network_access_permitted") is not False,
            authorization.get("credentials_permitted") is not False,
            authorization.get("customer_data_permitted") is not False,
            authorization.get("external_spend") != 0,
            authorization.get("measured_execution_authorized") is not False,
        )
    ):
        raise RuntimeError("authorization safety envelope drifted")

    expires_at = parse_timestamp(authorization.get("expires_at"))
    remaining_minutes = int(
        (expires_at - datetime.now(UTC)).total_seconds() // 60
    )
    if remaining_minutes < MINIMUM_LAUNCH_WINDOW_MINUTES:
        raise RuntimeError(
            "authorization has insufficient time remaining for a cold launch"
        )

    entries = dataset_manifest.get("entries")
    if not isinstance(entries, list) or len(entries) != 3:
        raise RuntimeError("dataset manifest entry set drifted")
    entries_by_role = {
        str(entry.get("role")): entry
        for entry in entries
        if isinstance(entry, dict)
    }
    expected_mounts = {
        "harness_source": EXPECTED_HARNESS_SOURCE.as_posix(),
        "model_artifacts": EXPECTED_MODEL_SNAPSHOT.as_posix(),
    }
    observed_mounts = {
        role: str(entries_by_role.get(role, {}).get("mounted_path"))
        for role in expected_mounts
    }
    if observed_mounts != expected_mounts:
        raise RuntimeError("dataset manifest static mount bindings drifted")
    runtime_entry = entries_by_role.get("vllm_runtime")
    if not isinstance(runtime_entry, dict):
        raise RuntimeError("dataset manifest CUDA 12.9 runtime entry is missing")
    expected_runtime = {
        "artifact_format": "python_wheelhouse_directory",
        "mounted_path": None,
        "sha256": EXPECTED_RUNTIME_SHA256_MANIFEST_SHA256,
        "runtime_output_directory": EXPECTED_RUNTIME_OUTPUT_DIRECTORY,
        "resolution_lock_sha256": EXPECTED_RUNTIME_RESOLUTION_LOCK_SHA256,
        "runtime_manifest_sha256": EXPECTED_RUNTIME_MANIFEST_SHA256,
        "sha256_manifest_sha256": EXPECTED_RUNTIME_SHA256_MANIFEST_SHA256,
        "materialization_receipt_sha256": (
            EXPECTED_RUNTIME_MATERIALIZATION_RECEIPT_SHA256
        ),
        "package_count": EXPECTED_RUNTIME_PACKAGE_COUNT,
    }
    if any(runtime_entry.get(key) != value for key, value in expected_runtime.items()):
        raise RuntimeError("dataset manifest CUDA 12.9 runtime authority drifted")

    stage = "reviewed_core_execution"

    reviewed_core = base64.b64decode(
        REVIEWED_CORE_B64.encode("ascii"),
        validate=True,
    )
    if sha256_bytes(reviewed_core) != EXPECTED_REVIEWED_CORE_SHA256:
        raise RuntimeError("reviewed qualification core identity drifted")

    os.environ["AURAGATEWAY_QUALIFICATION_AUTHORIZATION"] = str(
        authorization_path
    )
    os.environ["AURAGATEWAY_QUALIFICATION_DATASET_MANIFEST"] = str(
        dataset_manifest_path
    )

    execution_namespace: dict[str, object] = {}
    exec(
        compile(
            reviewed_core.decode("utf-8"),
            "auragateway_reviewed_qualification_core_v1.py",
            "exec",
        ),
        execution_namespace,
    )

    summary = execution_namespace.get("summary")
    if not isinstance(summary, dict):
        raise RuntimeError("qualification execution did not return a summary")

    expected_summary = {
        "model_request_count": 6,
        "runtime_evidence_generated": True,
        "runtime_evidence_count": 8,
        "environment_qualified": True,
        "measured_execution_authorized": False,
        "external_spend": 0,
        "next_gate": (
            "full_abc_local_full_run_environment_qualification_evidence_review"
        ),
    }
    for key, expected in expected_summary.items():
        if summary.get(key) != expected:
            raise RuntimeError(
                f"qualification summary drifted: {key}"
            )

    stage = "success_evidence_packaging"

    repo_root_raw = os.environ.get("AURAGATEWAY_REPO_ROOT")
    if repo_root_raw is None:
        raise RuntimeError("reviewed core did not bind AURAGATEWAY_REPO_ROOT")
    repo_root = Path(repo_root_raw).resolve()
    if repo_root != MATERIALIZED_HARNESS_ROOT.resolve():
        raise RuntimeError("reviewed core repository root drifted")

    evidence_files: list[tuple[str, Path]] = []
    evidence_checksums: dict[str, dict[str, object]] = {}
    for relative_path in RUNTIME_EVIDENCE_PATHS:
        path = repo_root / relative_path
        validate_regular_file(path, maximum_bytes=512 * 1024)
        evidence_files.append((Path(relative_path).name, path))
        evidence_checksums[Path(relative_path).name] = {
            "sha256": file_sha256(path),
            "size_bytes": path.stat().st_size,
        }

    launcher_summary = {
        "schema_version": "1.0.0",
        "status": "QUALIFIED",
        "notebook_name": NOTEBOOK_NAME,
        "captured_at": datetime.now(UTC).replace(microsecond=0).isoformat(),
        "reviewed_core_sha256": EXPECTED_REVIEWED_CORE_SHA256,
        "source_main_merge_commit": EXPECTED_SOURCE_MAIN_MERGE_COMMIT,
        "authorization_source_main_merge_commit": (
            authorization_source_main_merge_commit
        ),
        "authorization_file_sha256": file_sha256(authorization_path),
        "dataset_manifest_file_sha256": file_sha256(dataset_manifest_path),
        "remaining_minutes_at_launch": remaining_minutes,
        "summary": summary,
        "benchmark_trajectory_requests_permitted": 0,
        "customer_data_used": False,
        "credentials_used": False,
        "provider_calls_performed": False,
        "external_spend": 0,
    }

    if EVIDENCE_ZIP_PATH.exists():
        raise RuntimeError("qualification evidence ZIP appeared unexpectedly")

    with zipfile.ZipFile(
        EVIDENCE_ZIP_PATH,
        mode="x",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for archive_name, path in sorted(evidence_files):
            archive.write(path, arcname=archive_name)
        archive.writestr(
            "evidence_bundle_sha256.json",
            canonical_json(evidence_checksums),
        )
        archive.writestr(
            "launcher_summary.json",
            canonical_json(launcher_summary),
        )

    zip_size = EVIDENCE_ZIP_PATH.stat().st_size
    if zip_size > MAXIMUM_EVIDENCE_ZIP_BYTES:
        EVIDENCE_ZIP_PATH.unlink(missing_ok=True)
        raise RuntimeError("qualification evidence ZIP exceeded the size budget")

    print(f"artifact={EVIDENCE_ZIP_PATH}")
    print(f"size_bytes={zip_size}")
    print(f"sha256={file_sha256(EVIDENCE_ZIP_PATH)}")
    print("qualification_status=QUALIFIED")
    print("runtime_evidence_count=8")
    print("model_request_count=6")
    print("benchmark_trajectory_requests_permitted=0")
    print("upload_only_this_file=true")

except BaseException as error:
    write_failure_bundle(error)
    raise
